## Imports

In [1]:
import json
import faiss
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer

e:\Anaconda3\envs\voiceenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths and configuration

In [2]:
# Adjust this path if your Phase 3 folders are located elsewhere

CHUNK_DIR = Path("chunked_data")

INDEX_FILE = CHUNK_DIR / "faiss_index.bin"
METADATA_FILE = CHUNK_DIR / "embedding_metadata.json"

MODEL_NAME = "all-MiniLM-L6-v2"

TOP_K = 5

print("Index:", INDEX_FILE)
print("Metadata:", METADATA_FILE)

Index: chunked_data\faiss_index.bin
Metadata: chunked_data\embedding_metadata.json


## Load the saved FAISS index

In [3]:
index = faiss.read_index(
    str(INDEX_FILE)
)

print("FAISS index loaded.")
print("Vectors:", index.ntotal)
print("Dimension:", index.d)

FAISS index loaded.
Vectors: 96
Dimension: 384


## Load chunk metadata

In [4]:
with open(
    METADATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    chunks = json.load(f)

print("Chunks loaded:", len(chunks))

Chunks loaded: 96


## Load embedding model

In [5]:
embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1823.93it/s]


Embedding model loaded.


## Retrieval function

This is intentionally the same retrieval logic you used in your existing notebook.

In [6]:
def retrieve_chunks(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        result = chunks[idx].copy()

        result["similarity_score"] = float(score)

        results.append(result)

    return results

## Load your 116 short queries

In [7]:
evaluation_queries = [
    # =====================================================
    # VACATION / LEAVE (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "How much vacation time do employees get?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "Can I take leave after having a baby?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What holidays does the company observe?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "How many days of PTO do I earn per month?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "I'm not feeling well today, how does sick leave work?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "Is Thanksgiving a day off at Clef?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "My partner and I are adopting a child — what leave am I entitled to?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "Do fathers get parental leave too or just mothers?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What happens if a holiday falls on a weekend?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "I need some time away, what are my options?",
        "expected_document": "Vacation and Sick Leave.md"
    },

    # =====================================================
    # SABBATICAL (Direct, Specific Numbers, Paraphrased)
    # =====================================================
    {
        "query": "How long do I have to work before I'm eligible for a sabbatical?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "What is the duration of the sabbatical at Clef?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "Can I use my sabbatical to start a side business?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "Do I still accrue vacation days while on sabbatical?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "I've been here 5 years, what's the deal with the long break?",
        "expected_document": "Sabbatical.md"
    },

    # =====================================================
    # CONTINUING EDUCATION (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "What is the annual learning budget for employees?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Can I use company money to attend a conference?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Does Clef pay for online courses and books?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "How much does Clef reimburse for speaking at events?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Is there a mentorship program at Clef?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I joined in June, is my learning budget still $4,000?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "How many hours per week can I spend on learning projects?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Does the learning budget roll over to the next year?",
        "expected_document": "Continuing Education.md"
    },

    # =====================================================
    # HEALTHCARE & INSURANCE (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What health insurance does Clef offer?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "Does the company cover dental and vision?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "What percentage of health insurance does Clef pay for dependents?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "I already have insurance through my spouse, can I opt out?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "When does my health coverage start after joining?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "Is there life insurance or disability coverage?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },

    # =====================================================
    # SALARY & EQUITY (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "How is salary determined at Clef?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "What's the salary for a technical employee with more than 5 years experience?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "How many stock options do new employees receive?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "What is the equity vesting schedule?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "Can I trade $5k of salary for more equity?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "Why doesn't Clef negotiate salaries individually?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "How much do the founders make?",
        "expected_document": "Salary and Equity Compensation.md"
    },

    # =====================================================
    # REFERRAL BONUSES (Direct, Specific Numbers, Paraphrased)
    # =====================================================
    {
        "query": "How much is the referral bonus?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "If I refer someone who gets hired, when do I get paid?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "What's the process for referring a friend to work at Clef?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "Are founders eligible for the referral bonus?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "How long does a referred candidate have to be hired within?",
        "expected_document": "Referral Bonuses.md"
    },

    # =====================================================
    # OTHER PROTECTED ABSENCES (Direct, Vague, Related Policies)
    # =====================================================
    {
        "query": "How many days of bereavement leave can I take?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I got called for jury duty, am I still paid?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "My mother-in-law passed away, can I take time off?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "What is pregnancy disability leave and how long is it?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I have a family emergency, what should I do?",
        "expected_document": "Other Protected Absences.md"
    },

    # =====================================================
    # WORKING REMOTELY (Direct, Paraphrased, Employee Terminology)
    # =====================================================
    {
        "query": "What's Clef's policy on working from home?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Can I work remotely for a whole week?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Do I need my manager's approval to work remotely for an extended period?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Does Clef pay for coworking spaces?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "What happens if I'm not performing well while working remotely?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "How far in advance do I need to notify the team about extended remote work?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I want to work from my partner's place in another city for a few days — what do I need to do?",
        "expected_document": "Working Remotely.md"
    },

    # =====================================================
    # EMPLOYEE PRIVACY (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "Can the company read my emails?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Does Clef monitor internet usage?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Can my manager search my desk or laptop?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Is it okay to send personal emails from my work account?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "What should I do if I think my computer has a virus?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Can I share my work email password with a coworker?",
        "expected_document": "Employee Privacy.md"
    },

    # =====================================================
    # CODE OF CONDUCT (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What behavior is not tolerated in Clef community spaces?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Someone is being rude in the Clef Slack channel — what should I do?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Does the code of conduct apply to Twitter and Facebook?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Who do I contact about harassment in a Clef community?",
        "expected_document": "Code of Conduct in the Community.md"
    },

    # =====================================================
    # COMPLAINT POLICY (Direct, Vague, Related Policies)
    # =====================================================
    {
        "query": "How do I file a workplace complaint at Clef?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "Will I face retaliation for reporting a problem?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "What happens after I submit a complaint?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "What if my complaint is about one of the founders?",
        "expected_document": "Complaint Policy.md"
    },

    # =====================================================
    # DRUG & ALCOHOL POLICY (Direct, Vague)
    # =====================================================
    {
        "query": "Is alcohol allowed in the office?",
        "expected_document": "Drug and Alcohol Policy.md"
    },
    {
        "query": "What is Clef's policy on recreational drug use?",
        "expected_document": "Drug and Alcohol Policy.md"
    },
    {
        "query": "Can we have beer at a company celebration?",
        "expected_document": "Drug and Alcohol Policy.md"
    },

    # =====================================================
    # AT-WILL EMPLOYMENT (Direct, Paraphrased)
    # =====================================================
    {
        "query": "Can Clef fire me without a reason?",
        "expected_document": "At-Will Employment.md"
    },
    {
        "query": "What does at-will employment mean at Clef?",
        "expected_document": "At-Will Employment.md"
    },
    {
        "query": "Who can change my at-will employment status?",
        "expected_document": "At-Will Employment.md"
    },

    # =====================================================
    # EQUAL OPPORTUNITY EMPLOYMENT (Direct, Related Policies)
    # =====================================================
    {
        "query": "Does Clef discriminate based on gender or race?",
        "expected_document": "Equal Opportunity Employment.md"
    },
    {
        "query": "What is Clef's stance on diversity in hiring?",
        "expected_document": "Equal Opportunity Employment.md"
    },

    # =====================================================
    # CLEF VALUES (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What are Clef's core values?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "What does 'be better today than yesterday' mean?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "How does Clef think about inclusion?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "Why does Clef emphasize trust?",
        "expected_document": "Clef Values.md"
    },

    # =====================================================
    # MISSION STATEMENT (Direct, Vague)
    # =====================================================
    {
        "query": "What is Clef's mission?",
        "expected_document": "Mission Statement.md"
    },
    {
        "query": "What problem is Clef trying to solve?",
        "expected_document": "Mission Statement.md"
    },

    # =====================================================
    # ONBOARDING / WELCOME (Direct, Employee Terminology, Vague)
    # =====================================================
    {
        "query": "What should I expect on my first day at Clef?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "What are the typical working hours at Clef?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "How does Clef celebrate a new employee's first day?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "I just got hired, where do I start?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "What is the goal for my first day at work?",
        "expected_document": "Welcome to Clef.md"
    },

    # =====================================================
    # HANDBOOK INTRODUCTION (Direct)
    # =====================================================
    {
        "query": "Who is the CEO of Clef?",
        "expected_document": "Handbook Introduction.md"
    },
    {
        "query": "What is the purpose of the employee handbook?",
        "expected_document": "Handbook Introduction.md"
    },

    # =====================================================
    # ONE ON ONES (Direct, Paraphrased, Employee Terminology)
    # =====================================================
    {
        "query": "How often are one-on-one meetings held?",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "Who sets the agenda for 1:1 meetings?",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "What's the minimum duration of a one on one?",
        "expected_document": "One on Ones.md"
    },

    # =====================================================
    # OBJECTIVES AND KEY RESULTS (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "How are OKRs scored at Clef?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "How often do we set OKRs?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "Are OKRs used for performance reviews?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "What is a good OKR score?",
        "expected_document": "Objectives and Key Results.md"
    },

    # =====================================================
    # COMMUNICATION & TRANSPARENCY (Direct, Paraphrased)
    # =====================================================
    {
        "query": "What happens on Fridays in terms of team updates?",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "How should I use Slack when working remotely?",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "Should conversations happen in public or private Slack channels?",
        "expected_document": "Communication and Transparency.md"
    },

    # =====================================================
    # DIRECT REPORTS (Direct)
    # =====================================================
    {
        "query": "Who do employees report to at Clef?",
        "expected_document": "Direct Reports.md"
    },

    # =====================================================
    # PRODUCT MANIFESTO (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What is Clef's core product value?",
        "expected_document": "Product Manifesto.md"
    },
    {
        "query": "Why do people love using Clef over other login solutions?",
        "expected_document": "Product Manifesto.md"
    },

    # =====================================================
    # OPERATIONS: BUDGETING, HACK WEEKS, SHARING FILES, EFFECTIVE MEETINGS
    # =====================================================
    {
        "query": "How is company spending organized at Clef?",
        "expected_document": "Budgeting.md"
    },
    {
        "query": "What are the budget categories at Clef?",
        "expected_document": "Budgeting.md"
    },
    {
        "query": "What is a hack week and when does it happen?",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "Can I work on personal projects during hack week?",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "How should files and projects be organized at Clef?",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "What are the base directories every Clef employee should have?",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "What are the meeting time requirements at Clef?",
        "expected_document": "Effective Meetings.md"
    },
    {
        "query": "Do meetings need to have a video call option?",
        "expected_document": "Effective Meetings.md"
    },

    # =====================================================
    # POLICY CHANGES (Direct, Paraphrased)
    # =====================================================
    {
        "query": "How are policy changes proposed and adopted at Clef?",
        "expected_document": "Policy Changes.md"
    },
    {
        "query": "What tools does Clef use for policy discussions?",
        "expected_document": "Policy Changes.md"
    },

    # =====================================================
    # CROSS-DOCUMENT / SEMANTICALLY SIMILAR DOCUMENT QUERIES
    # These are tricky — two docs could plausibly match
    # =====================================================
    {
        "query": "What kind of time off can I get after becoming a new parent?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "How does taking parental leave affect my sabbatical eligibility?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What's the difference between sick leave and vacation?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "If something bad happens at work who should I talk to — my manager or HR?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "Does the code of conduct apply inside the office or only online?",
        "expected_document": "Code of Conduct in the Community.md"
    },
]


## Load your 41 long queries

In [8]:
evaluation_queries_long = [
    # =====================================================
    # VACATION / LEAVE — Long-form queries
    # =====================================================
    {
        "query": "I've been working at Clef for about six months now and I'm wondering how many vacation days I've accumulated so far and whether unused sick days carry over to the next year",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "My spouse and I are expecting a baby in a few months and I want to understand the full maternity and paternity leave policy including how it interacts with California state disability programs",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "I recently adopted a child and I'm trying to figure out within what timeframe I need to take my new parent leave and whether I need to give any advance notice to my team",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "I have a chronic illness that sometimes makes it hard to come into the office regularly — is there any policy around flexible work arrangements or disability support for someone in my situation",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "If I take the full twelve weeks of new parent leave does that time count toward my five year requirement for becoming eligible for a sabbatical or does it pause the clock",
        "expected_document": "New Parent Leave.md"
    },

    # =====================================================
    # SABBATICAL — Long-form queries
    # =====================================================
    {
        "query": "I've been at Clef for almost five years and I'm interested in taking a sabbatical to volunteer at a nonprofit — do I need to present something to the team when I return and how far in advance should I notify everyone",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "I'm worried that if I take a three month sabbatical I might lose my accrued sick days — can you explain whether paid time off continues to accumulate during the sabbatical period or if it freezes",
        "expected_document": "Sabbatical.md"
    },

    # =====================================================
    # CONTINUING EDUCATION — Long-form queries
    # =====================================================
    {
        "query": "I want to attend a machine learning conference in Europe next quarter — would the flight and hotel costs come out of my learning budget or is there a separate travel budget for professional development events",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I've been invited to speak at a cybersecurity conference and I'm wondering how much Clef will cover for travel expenses and whether this comes from the same pool as the annual learning budget or a different one",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I started working at Clef in July of this year and I want to know if my learning budget is the full four thousand dollars or if it is reduced since I joined after the midpoint of the calendar year",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Can I use a portion of my work hours during the week to study for a certification that is related to my role at Clef and if so is there a limit on how many hours I can dedicate to that",
        "expected_document": "Continuing Education.md"
    },

    # =====================================================
    # HEALTHCARE & INSURANCE — Long-form queries
    # =====================================================
    {
        "query": "My wife already has health insurance through her employer and I don't need Clef's medical plan — is there any financial incentive or allowance if I choose to waive my medical coverage through TriNet",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "I'm starting at Clef next month and I want to add my two kids to my health insurance plan — what percentage of the dependent coverage does the company contribute and when does the coverage actually begin",
        "expected_document": "Healthcare and Disability Insurance.md"
    },

    # =====================================================
    # SALARY & EQUITY — Long-form queries
    # =====================================================
    {
        "query": "I'm a software engineer with seven years of experience and I'd like to understand what my starting salary would be at Clef and whether there is room for individual negotiation beyond the standard rubric",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "I noticed the equity vesting schedule at Clef is six years instead of the typical four — can you explain the reasoning behind that and how many stock options a new employee typically receives",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "If I choose to take the five thousand dollar salary reduction in exchange for additional equity how many extra stock options would I receive and what total percentage of the company would that represent",
        "expected_document": "Salary and Equity Compensation.md"
    },

    # =====================================================
    # REFERRAL BONUSES — Long-form queries
    # =====================================================
    {
        "query": "I referred a friend to Clef about four months ago and she just got an offer — do I qualify for the referral bonus and when exactly would I expect to receive the payment after she starts",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "Two people on the team both claim they referred the same candidate independently — how does the company determine who gets the five thousand dollar referral bonus in a situation like this",
        "expected_document": "Referral Bonuses.md"
    },

    # =====================================================
    # WORKING REMOTELY — Long-form queries
    # =====================================================
    {
        "query": "I'm planning to work from my parents house in another state for about ten days next month — do I need to get my manager's approval beforehand and how much advance notice should I give the rest of the team",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I've been working remotely a couple days a week but my manager says my productivity has dropped — can they revoke my remote work privileges entirely and what process has to happen before that decision is made",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "When I'm working from home I know I should be available on Slack but I'm unclear about the exact expectations — is there a specific set of hours where I need to be reachable for meetings and collaboration",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I want to work from a coffee shop while traveling but I'm concerned about the internet connection requirements — what are the specific prerequisites Clef expects me to have in place before working from somewhere irregular",
        "expected_document": "Working Remotely.md"
    },

    # =====================================================
    # EMPLOYEE PRIVACY — Long-form queries
    # =====================================================
    {
        "query": "I sometimes send personal emails from my Clef email address and I want to know if the company has the ability to read those messages even if I mark them as private or personal in the subject line",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "I keep a personal journal on my work laptop and I want to know if management has the right to access files stored on company devices or if my personal data on company property is protected",
        "expected_document": "Employee Privacy.md"
    },

    # =====================================================
    # CODE OF CONDUCT — Long-form queries
    # =====================================================
    {
        "query": "A community member made an offensive joke in the Clef Slack channel and someone complained about it — what does the code of conduct say about jokes and what are the consequences for the person who made the comment",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "I witnessed someone being harassed at a Clef-hosted event last week and I want to report it — who exactly should I contact and does it matter if the harassment was not directed at me personally",
        "expected_document": "Code of Conduct in the Community.md"
    },

    # =====================================================
    # COMPLAINT POLICY — Long-form queries
    # =====================================================
    {
        "query": "I want to file a complaint about something that happened at work but I'm afraid my manager might treat me differently afterward — does Clef have a policy against retaliation for employees who report issues",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "If my complaint involves one of the founders of the company how is the investigation handled differently and is there a possibility that an outside investigator would be brought in to ensure impartiality",
        "expected_document": "Complaint Policy.md"
    },

    # =====================================================
    # OKRs — Long-form queries
    # =====================================================
    {
        "query": "I'm new to the OKR system and I don't understand how scoring works at the end of the quarter — if I score a perfect ten on all my key results does that mean I set my goals too low or is that considered ideal",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "I want to set a key result that involves delivering a project by a certain date — how should I structure the scoring so that points are deducted fairly if the project ends up being delivered late",
        "expected_document": "Objectives and Key Results.md"
    },

    # =====================================================
    # ONE ON ONES — Long-form queries
    # =====================================================
    {
        "query": "I have a problem with a coworker and I want to bring it up during my one on one meeting — will my manager step in to address it directly or do they need my permission before talking to that person about it",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "My one on one meetings with my manager feel more like status updates than meaningful conversations — what does Clef recommend regarding the agenda and tone of these meetings to make them more productive",
        "expected_document": "One on Ones.md"
    },

    # =====================================================
    # COMMUNICATION & TRANSPARENCY — Long-form queries
    # =====================================================
    {
        "query": "When I set my Slack status to Do Not Disturb and the indicator shows green what does that signal to my teammates — should they expect a response immediately or should they wait until my focus time is over",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "I've noticed that a lot of important decisions are being discussed in private Slack channels and I feel out of the loop — what is Clef's official guideline on using public versus private channels for team conversations",
        "expected_document": "Communication and Transparency.md"
    },

    # =====================================================
    # OPERATIONS — Long-form queries
    # =====================================================
    {
        "query": "I have an idea for a hack week project that would require three other people to help me build it — how do I pitch this project to the team and are there any restrictions on what we can work on during that week",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "When a project is completed and no longer active how should I archive the project folder and what naming convention does Clef use so that old projects are easy to find later on",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "I need to schedule a meeting with a remote colleague but they are in a different time zone — what are the core hours during which all Clef employees are expected to be available for face to face meetings",
        "expected_document": "Effective Meetings.md"
    },

    # =====================================================
    # POLICY CHANGES — Long-form queries
    # =====================================================
    {
        "query": "I disagree with a recent policy change that was merged into the handbook and I want to voice my concerns — what is the proper channel for giving feedback and will there be a team discussion about it before it takes effect",
        "expected_document": "Policy Changes.md"
    },
    {
        "query": "I heard that employees need to sign an acknowledgement every time the handbook is updated — how often do these updates get adopted and is there a formal review process before changes become official policy",
        "expected_document": "Policy Changes.md"
    },

    # =====================================================
    # CROSS-DOCUMENT / TRICKY — Long-form queries
    # =====================================================
    {
        "query": "I'm pregnant and dealing with severe morning sickness that makes it hard to come to work every day — can I take intermittent pregnancy disability leave while still keeping my job and benefits at Clef",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I want to understand how Clef avoids salary bias during the hiring process — is there a standard pay scale for all roles or does compensation depend on individual negotiation skills and previous salary history",
        "expected_document": "Salary and Equity Compensation.md"
    },
]


## Run all 116 queries

In [9]:
short_results = []

for item in evaluation_queries:

    query = item["query"]
    expected = item["expected_document"]

    retrieved = retrieve_chunks(
        query,
        top_k=TOP_K
    )

    query_result = {
        "query": query,
        "expected_document": expected,
        "results": []
    }

    for rank, result in enumerate(
        retrieved,
        start=1
    ):

        query_result["results"].append({
            "rank": rank,
            "document": result["document"],
            "section_title": result["section_title"],
            "section_path": result["section_path"],
            "similarity_score": result["similarity_score"],
            "word_count": result["word_count"],
            "content": result["content"]
        })

    short_results.append(query_result)

print(
    "Completed short-query retrieval:",
    len(short_results)
)

Completed short-query retrieval: 116


This gives you all Top-5 results, not just Rank #1. That's important because later we can investigate why Rank #1 failed.

## Save the 116-query results

In [ ]:
OUTPUT_DIR = Path("failure_analysis_results")
OUTPUT_DIR.mkdir(exist_ok=True)

short_output = (
    OUTPUT_DIR /
    "short_queries_116_retrieval_outputs.json"
)

with open(
    short_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        short_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:")
print(short_output)

Saved:
failure_analysis_results\short_queries_117_retrieval_outputs.json


## Run all 41 long queries

In [11]:
long_results = []

for item in evaluation_queries_long:

    query = item["query"]
    expected = item["expected_document"]

    retrieved = retrieve_chunks(
        query,
        top_k=TOP_K
    )

    query_result = {
        "query": query,
        "expected_document": expected,
        "results": []
    }

    for rank, result in enumerate(
        retrieved,
        start=1
    ):

        query_result["results"].append({
            "rank": rank,
            "document": result["document"],
            "section_title": result["section_title"],
            "section_path": result["section_path"],
            "similarity_score": result["similarity_score"],
            "word_count": result["word_count"],
            "content": result["content"]
        })

    long_results.append(query_result)

print(
    "Completed long-query retrieval:",
    len(long_results)
)

Completed long-query retrieval: 41


## Save the 41-query results

In [12]:
long_output = (
    OUTPUT_DIR /
    "long_queries_41_retrieval_outputs.json"
)

with open(
    long_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        long_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:")
print(long_output)

Saved:
failure_analysis_results\long_queries_41_retrieval_outputs.json


## Final validation

In [13]:
print("FAILURE ANALYSIS DATASET")
print("=" * 50)

print(
    "Short queries:",
    len(short_results)
)

print(
    "Long queries:",
    len(long_results)
)

print(
    "Short output exists:",
    short_output.exists()
)

print(
    "Long output exists:",
    long_output.exists()
)

print(
    "Top-K stored per query:",
    TOP_K
)

FAILURE ANALYSIS DATASET
Short queries: 116
Long queries: 41
Short output exists: True
Long output exists: True
Top-K stored per query: 5
